## LOCAL WINDOWS SETUP

### Before running:
1. Install ffmpeg: `choco install ffmpeg` (or download from ffmpeg.org)
2. Install Python 3.10+ with pip
3. Create virtual environment: `python -m venv venv` then `venv\Scripts\activate`
4. Adjust paths if needed (defaults: `./datasets`, `./cdt_dataset`, `./cdt/`)

### GPU Setup (Optional but recommended):
- NVIDIA GPU: Install CUDA Toolkit 12.1 + cuDNN (torch will auto-detect)
- CPU-only: Code still works, just slower training


# Crowd Digital Twin — Complete Colab Notebook
**Team 28 | PES University | UE23CS320A Capstone Phase 2**  
Guide: Dr. Richa Sharma

### Pipeline
```
Video/CCTV → VideoInputHandler → DataFusionLayer (3 streams)
→ YOLOv8n → ByteTrack → ConfidenceEstimator
→ DigitalTwinEngine → DensityFlowEstimator
→ TrendPredictor → RiskEstimator → AlertEngine
→ WebSocket → Dashboard
```

### Cells
1. Install dependencies
2. Download all datasets
3. Convert datasets to YOLO format
4. Fine-tune YOLOv8n on combined data
5. Evaluate on UCF-CC-50
6. Prepare video source
7. Upload project files
8. Start real-time server + ngrok
9. Monitor pipeline stats
10. Save results & evaluation metrics

### Save Model Weights and Datasets to Google Drive

To ensure your trained model and downloaded datasets persist across Colab sessions, you should save them to Google Drive. This involves mounting your Drive and then copying the necessary directories and files.

In [1]:
# ── Load saved model (future sessions — skip Cell 4 entirely) ─────────────────
from google.colab import drive
import shutil, os
from ultralytics import YOLO

drive.mount('/content/drive')

src = '/content/drive/MyDrive/cdt_project_data/cdt/weights/yolov8n_cdt.pt'
dst = '/content/cdt/weights/yolov8n_cdt.pt'

os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(src, dst)

model = YOLO(dst)
print('✅ Model loaded — skip Cell 4, proceed from Cell 5')

ModuleNotFoundError: No module named 'google'

In [ ]:
# ── Cell 1: Install all dependencies (LOCAL WINDOWS) ──────────────────────────
import subprocess, sys, os

print("Installing dependencies...")

packages = [
    'fastapi==0.111.0',
    'uvicorn[standard]==0.30.1',
    'websockets==12.0',
    'ultralytics==8.2.0',
    'supervision==0.21.0',
    'opencv-python==4.10.0.84',  # Use opencv-python (not headless) for Windows
    'numpy==1.26.4',
    'scipy==1.13.0',
    'pandas==2.2.2',
    'torch',  # Will auto-detect CUDA if available
    'torchvision',
]

for p in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=True)

import torch
print(f'✅ CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('✅ All dependencies installed')

Cleaning environment...


In [ ]:
# ── Cell 2: Download all datasets ────────────────────────────────────────────
import os
import subprocess
import shutil # Added for robust directory removal
os.makedirs('/content/datasets', exist_ok=True)

# MOT17 (~1 GB) — person detection + tracking
if not os.path.exists('/content/datasets/MOT17'):
    print('Downloading MOT17...')
    os.system('wget -q -nc https://motchallenge.net/data/MOT17.zip -O /content/datasets/MOT17.zip')
    os.system('unzip -q /content/datasets/MOT17.zip -d /content/datasets/')
    print('✅ MOT17 downloaded')
elif os.path.exists('/content/datasets/MOT17') and not os.path.isdir('/content/datasets/MOT17'):
    # Handle case where MOT17 exists but is not a directory (e.g., failed partial download)
    print('⚠️ MOT17 found but not a directory. Cleaning up and re-downloading...')
    os.remove('/content/datasets/MOT17') # Remove the file if it's not a directory
    os.system('wget -q -nc https://motchallenge.net/data/MOT17.zip -O /content/datasets/MOT17.zip')
    os.system('unzip -q /content/datasets/MOT17.zip -d /content/datasets/')
    print('✅ MOT17 re-downloaded')
else:
    print('✅ MOT17 already present')

# UCF-CC-50 (~100 MB) — dense crowd counting
# The UCF_CC_50.zip extracts files directly into the target directory, not into a subfolder named UCF_CC_50.
# We check for a representative file (e.g., 1.jpg) to confirm its presence.
ucf_img_check_path = '/content/datasets/1.jpg'
ucf_zip_path = '/content/datasets/UCF_CC_50.zip'
ucf_extracted_dir = '/content/datasets/UCF_CC_50' # Possible extraction directory if zip contains it

# Check if 1.jpg is present directly in /content/datasets or inside UCF_CC_50 subfolder
if not os.path.exists(ucf_img_check_path) and not os.path.exists(os.path.join(ucf_extracted_dir, '1.jpg')):
    print('Downloading UCF-CC-50...')
    # Ensure previous potentially corrupted zip/extracted files are removed and force a fresh download
    if os.path.exists(ucf_zip_path):
        os.remove(ucf_zip_path)
        print('  Removed old UCF_CC_50.zip')
    if os.path.exists(ucf_extracted_dir):
        shutil.rmtree(ucf_extracted_dir, ignore_errors=True) # Remove previously extracted directory if it exists

    # Download and extract UCF-CC-50
    wget_command = f"wget -q 'https://www.crcv.ucf.edu/data/ucf-cc-50/UCF_CC_50.zip' -O {ucf_zip_path}"
    os.system(wget_command)
    print(f'  Downloaded UCF_CC_50.zip. Checking contents...')
    try:
        # List contents of the zip file to debug, but don't fail if zip is bad
        result = subprocess.run(['unzip', '-l', ucf_zip_path], capture_output=True, text=True, check=True, timeout=10)
        # Show last few lines of the unzip -l output
        print('  ' + '\n  '.join(result.stdout.splitlines()[-5:]))
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        print(f'  ⚠️ Could not list contents of zip (possibly corrupted): {e}')

    os.system(f'unzip -q {ucf_zip_path} -d /content/datasets/')
    print('✅ UCF-CC-50 downloaded and extracted.')

    # Re-check for 1.jpg after extraction
    if os.path.exists(ucf_img_check_path):
        print('  Found 1.jpg directly in /content/datasets/')
    elif os.path.exists(os.path.join(ucf_extracted_dir, '1.jpg')):
        print('  Found 1.jpg in /content/datasets/UCF_CC_50/')
    else:
        print('  ⚠️ Warning: 1.jpg not found in expected locations after extraction.')
else:
    print('✅ UCF-CC-50 already present')


# ETH/UCY (~5 MB) — pedestrian trajectories
# Ensure clean download and extraction for ETH/UCY
if not os.path.exists('/content/datasets/eth_ucy'):
    print('Downloading ETH/UCY...')
    os.system('rm -rf /content/trajnet_tmp') # Clean up previous partial clone
    os.system('rm -rf /content/datasets/eth_ucy') # Clean up previous partial copy
    os.system('git clone -q https://github.com/StanfordASL/Trajectron-plus-plus /content/trajnet_tmp')
    os.system('cp -r /content/trajnet_tmp/experiments/pedestrians/raw/ /content/datasets/eth_ucy/')
    os.system('rm -rf /content/trajnet_tmp')
    print('✅ ETH/UCY downloaded')
else:
    print('✅ ETH/UCY already present')

# Mall Dataset (~500 MB) — indoor CCTV
if not os.path.exists('/content/datasets/mall_dataset'):
    print('Downloading Mall Dataset...')
    os.system("wget -q 'http://personal.ie.cuhk.edu.hk/~ccloy/files/datasets/mall_dataset.zip' -O /content/datasets/mall.zip")
    os.system('unzip -q /content/datasets/mall.zip -d /content/datasets/')
    print('✅ Mall Dataset downloaded')
else:
    print('✅ Mall Dataset already present')

print('\nAll datasets ready:')
# Adjusted checks for existing datasets
for d in ['MOT17','eth_ucy','mall_dataset']:
    status = '✅' if os.path.exists(f'/content/datasets/{d}') else '❌'
    print(f'  {status} {d}')
# Specific check for UCF-CC-50
status_ucf = '✅' if os.path.exists(ucf_img_check_path) or os.path.exists(os.path.join(ucf_extracted_dir, '1.jpg')) else '❌'
print(f'  {status_ucf} UCF_CC_50')

# Added for debugging: list all extracted files recursively
print('\nDetailed contents of /content/datasets/ after extraction:')
os.system('ls -RF /content/datasets/')


In [ ]:
# ── Cell 3 UPDATED: Convert all datasets to YOLO format with MOT17 fix ────────
import subprocess, sys, os, shutil

# CRITICAL: Clean pip cache and force complete reinstall
print('🔧 Cleaning pip cache and forcing complete reinstall...')
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False, capture_output=True)

# Uninstall existing numpy/scipy completely
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'numpy', 'scipy', '-y', '-q'],
               check=False, capture_output=True)

# Fresh install with compatible versions
print('📦 Installing clean numpy==1.26.4 and scipy==1.13.0...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                'numpy==1.26.4', 'scipy==1.13.0', '-q'], check=True)
print('✅ Clean install complete.\n')

# NOW import (fresh Python context)
import cv2
import scipy.io
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.ndimage import gaussian_filter

os.makedirs('/content/cdt_dataset/images', exist_ok=True)
os.makedirs('/content/cdt_dataset/labels', exist_ok=True)
os.makedirs('/content/cdt_dataset/density', exist_ok=True)

# ── STEP 1: Inspect MOT17 data format ──────────────────────────────────────────
print('📋 Inspecting MOT17 data format...')
mot_root = Path('/content/datasets/MOT17/train')
gt_files = list(mot_root.glob('*/gt/gt.txt'))
if gt_files:
    sample_gt = pd.read_csv(str(gt_files[0]), header=None,
                             names=['frame','id','x','y','w','h','conf','cls','vis'])
    print(f"\n  Sample GT file: {gt_files[0].parts[-3]}")
    print(f"  Shape: {sample_gt.shape}")
    print(f"\n  First 5 rows:")
    print(sample_gt.head())
    print(f"\n  Unique classes: {sorted(sample_gt['cls'].unique())}")
    print(f"  Confidence range: [{sample_gt['conf'].min():.2f}, {sample_gt['conf'].max():.2f}]")
    print(f"  Visibility range: [{sample_gt['vis'].min()}, {sample_gt['vis'].max()}]")
else:
    print("  ⚠️  No MOT17 files found!")

# ── Converter A: MOT17 → YOLO (FIXED with flexible filtering) ──────────────────
def convert_mot17(mot_root_str, out_root):
    """
    MOT17 GT format: frame, id, x, y, w, h, conf, cls, vis
    - cls==1: pedestrian
    - conf: detection confidence (0-1 range)
    - vis: visibility (0-1 range)

    Strategy: Keep class==1 AND conf>0.5 to get high-confidence person detections
    """
    mot_root = Path(mot_root_str) # Convert input string to Path object
    count = 0
    skipped_seqs = 0
    total_boxes = 0
    kept_boxes = 0

    for gt_file in sorted(mot_root.glob('*/gt/gt.txt')):
        seq = gt_file.parts[-3]
        img_dir = gt_file.parent.parent / 'img1'

        if not img_dir.exists():
            print(f"    {seq}: image dir not found — skipping")
            skipped_seqs += 1
            continue

        gt = pd.read_csv(str(gt_file), header=None,
                          names=['frame','id','x','y','w','h','conf','cls','vis'])

        # FIXED filtering: class==1 (person) AND confidence > 0.5
        total_boxes += len(gt)
        gt_filtered = gt[(gt['cls']==1) & (gt['conf']>0.5)]
        kept_boxes += len(gt_filtered)

        if len(gt_filtered) == 0:
            print(f"    {seq}: no valid detections after filtering")
            skipped_seqs += 1
            continue

        out_img = Path(out_root) / 'images' / seq
        out_lbl = Path(out_root) / 'labels' / seq
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)

        seq_count = 0
        for fname in sorted(img_dir.glob('*.jpg')):
            fid = int(fname.stem)
            img = cv2.imread(str(fname))
            if img is None:
                continue

            H, W = img.shape[:2]
            rows = gt_filtered[gt_filtered['frame']==fid]

            if len(rows) > 0:
                with open(out_lbl / (fname.stem+'.txt'), 'w') as f:
                    for _, r in rows.iterrows():
                        cx = (r.x + r.w/2) / W
                        cy = (r.y + r.h/2) / H
                        f.write(f'0 {cx:.6f} {cy:.6f} {r.w/W:.6f} {r.h/H:.6f}\n')
                seq_count += 1

            shutil.copy(fname, out_img/fname.name)
            count += 1

        print(f"    {seq}: {seq_count} frames with detections")

    print(f"  MOT17: {count} total frames, {kept_boxes}/{total_boxes} boxes kept")
    if skipped_seqs > 0:
        print(f"         ({skipped_seqs} sequences skipped due to missing data)")

# ── Converter B: UCF-CC-50 → Density maps ─────────────────────────────────────
def convert_ucf_cc50(ucf_root, out_root):
    count = 0
    for i in range(1, 51):
        img_path = f'{ucf_root}/{i}.jpg'
        ann_path = f'{ucf_root}/{i}_ann.mat'
        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)
        ann = scipy.io.loadmat(ann_path)
        pts = ann['annPoints']
        h, w = img.shape[:2]
        dm = np.zeros((h, w), dtype=np.float32)

        for (x, y) in pts:
            xi = int(np.clip(x, 0, w-1))
            yi = int(np.clip(y, 0, h-1))
            dm[yi, xi] += 1.0

        dm = gaussian_filter(dm, sigma=15)
        np.save(f'{out_root}/density/ucf_{i:02d}.npy', dm)
        shutil.copy(img_path, f'{out_root}/images/ucf_{i:02d}.jpg')
        count += 1

    print(f"  UCF-CC-50: {count} density maps created")

# ── Converter C: Mall Dataset → YOLO ──────────────────────────────────────────
def convert_mall(mall_root, out_root):
    gt_path = f'{mall_root}/mall_gt.mat'
    img_dir = f'{mall_root}/frames'

    if not os.path.exists(gt_path):
        print(f"  Mall Dataset: {gt_path} not found — skipping")
        return

    out_img = Path(out_root) / 'images' / 'mall'
    out_lbl = Path(out_root) / 'labels' / 'mall'
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    gt = scipy.io.loadmat(gt_path)
    frames = gt['frame'][0]
    count = 0

    for i, fr in enumerate(frames):
        img_path = f'{img_dir}/seq_{i+1:06d}.jpg'
        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue

        H, W = img.shape[:2]
        positions = fr[0][0][0]

        with open(out_lbl / f'mall_{i:04d}.txt', 'w') as f:
            for (x, y) in positions:
                bw = 50 / W
                bh = 120 / H
                cx = x / W
                cy = (y + 60) / H
                f.write(f'0 {cx:.4f} {cy:.4f} {bw:.4f} {bh:.4f}\n')

        cv2.imwrite(str(out_img / f'mall_{i:04d}.jpg'), img)
        count += 1

    print(f"  Mall: {count} frames converted")

# ── RUN ALL CONVERTERS ────────────────────────────────────────────────────────
print('\n' + '='*70)
print('Running converters...')
print('='*70 + '\n')

convert_mot17('/content/datasets/MOT17/train', '/content/cdt_dataset')
convert_ucf_cc50('/content/datasets/UCF_CC_50', '/content/cdt_dataset')
convert_mall('/content/datasets/mall_dataset', '/content/cdt_dataset')

print('\n' + '='*70)
print('✅ All conversions complete!')
print('='*70)

# Summary statistics
print('\n📊 Dataset Summary:')
img_count = len(list(Path('/content/cdt_dataset/images').rglob('*.jpg')))
lbl_count = len(list(Path('/content/cdt_dataset/labels').rglob('*.txt')))
density_count = len(list(Path('/content/cdt_dataset/density').rglob('*.npy')))

print(f"  Images: {img_count}")
print(f"  Labels: {lbl_count}")
print(f"  Density maps: {density_count}")

In [ ]:
# ── Cell 4 UPDATED: Auto-detect datasets, sanitize labels, fine-tune YOLOv8n ──
import os
import glob
import shutil
from pathlib import Path
from ultralytics import YOLO
import torch

# ── PyTorch 2.4+ / Python 3.12 compatibility patch ───────────────────────────
def apply_torch_patch():
    current_load = torch.load
    if hasattr(current_load, '__name__') and current_load.__name__ == 'patched_torch_load':
        return
    root_load = current_load
    while hasattr(root_load, '__wrapped__'):
        root_load = root_load.__wrapped__
        if hasattr(root_load, '__name__') and root_load.__name__ == 'patched_torch_load':
            return

    def patched_torch_load(*args, **kwargs):
        if 'weights_only' not in kwargs:
            kwargs['weights_only'] = False
        return root_load(*args, **kwargs)

    torch.load = patched_torch_load
    torch._is_patched_for_ultralytics = True
    print('✅ PyTorch load patched (root-protected)')

apply_torch_patch()


# ── STEP 1: Auto-detect available image sequences ────────────────────────────
print('📋 Auto-detecting available image sequences...\n')

img_root = Path('/content/cdt_dataset/images')
lbl_root = Path('/content/cdt_dataset/labels')

available_sequences = sorted([d.name for d in img_root.iterdir() if d.is_dir()])

print(f'Found {len(available_sequences)} sequences:')
for seq in available_sequences:
    img_count = len(list((img_root / seq).glob('*.jpg')))
    print(f'  - {seq}: {img_count} images')

if not available_sequences:
    print('❌ No image sequences found!')
    exit(1)


# ── STEP 2: Sanitize label files (fix corrupt YOLO labels) ───────────────────
print('\n🔧 Sanitizing label files...\n')

def sanitize_label_file(label_path: Path) -> tuple[int, int]:
    """
    Clamp all bounding box values to [0, 1] and remove empty/malformed lines.
    Returns (valid_count, removed_count).
    """
    try:
        raw = label_path.read_text().strip()
    except Exception:
        return 0, 0

    if not raw:
        return 0, 0

    valid_lines = []
    removed = 0

    for line in raw.splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            removed += 1
            continue
        try:
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])
        except ValueError:
            removed += 1
            continue

        # Skip boxes that are entirely out of range or degenerate
        if w <= 0 or h <= 0:
            removed += 1
            continue

        # Clamp to valid YOLO range [0, 1]
        x = max(0.0, min(1.0, x))
        y = max(0.0, min(1.0, y))
        w = max(0.001, min(1.0, w))
        h = max(0.001, min(1.0, h))

        valid_lines.append(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    label_path.write_text('\n'.join(valid_lines))
    return len(valid_lines), removed

total_valid   = 0
total_removed = 0
seqs_with_labels = []

for seq in available_sequences:
    seq_lbl = lbl_root / seq
    if not seq_lbl.exists():
        continue

    seq_valid = seq_removed = 0
    for lf in seq_lbl.glob('*.txt'):
        v, r = sanitize_label_file(lf)
        seq_valid   += v
        seq_removed += r

    total_valid   += seq_valid
    total_removed += seq_removed
    seqs_with_labels.append(seq)

    status = f'{seq_valid} valid annotations'
    if seq_removed:
        status += f', {seq_removed} removed'
    print(f'  {seq}: {status}')

print(f'\n✅ Sanitization complete — {total_valid} valid, {total_removed} removed across all sequences')

if total_valid == 0:
    print('❌ No valid labels after sanitization — check your label conversion pipeline!')
    exit(1)


# ── STEP 3: Split into train / val ───────────────────────────────────────────
print('\n🔀 Splitting into train/val...\n')

mot_seqs  = [s for s in seqs_with_labels if 'MOT17' in s]
mall_seqs = [s for s in seqs_with_labels if 'mall'  in s]
ucf_seqs  = [s for s in available_sequences if 'ucf' in s]

print(f'MOT17 sequences (labeled): {len(mot_seqs)}')
print(f'Mall  sequences (labeled): {len(mall_seqs)}')
print(f'UCF   sequences (no YOLO labels — skipped): {len(ucf_seqs)}')

labeled_seqs = mot_seqs + mall_seqs

# 80/20 split — guarantee at least 1 val sequence
split_idx = max(1, int(len(labeled_seqs) * 0.8))
train_seqs = labeled_seqs[:split_idx]
val_seqs   = labeled_seqs[split_idx:]

# Edge case: only 1 sequence total
if not val_seqs and len(train_seqs) > 1:
    val_seqs = [train_seqs.pop()]

# Move mall to val if it would otherwise be the only val representative
# (different camera domain — better as held-out test)
if mall_seqs and all(s in train_seqs for s in mall_seqs):
    for s in mall_seqs:
        train_seqs.remove(s)
        val_seqs.append(s)
    print('\n⚠️  Mall moved to val-only (different camera domain)')

print(f'\nTrain sequences ({len(train_seqs)}):')
for s in train_seqs: print(f'  - {s}')
print(f'\nVal sequences ({len(val_seqs)}):')
for s in val_seqs:   print(f'  - {s}')


# ── STEP 4: Generate dataset.yaml ────────────────────────────────────────────
print('\n📝 Generating dataset.yaml...\n')

yaml_lines = ['path: /content/cdt_dataset\n', 'train:']
for s in train_seqs:
    yaml_lines.append(f'  - images/{s}')
yaml_lines.append('\nval:')
for s in val_seqs:
    yaml_lines.append(f'  - images/{s}')
yaml_lines += ['\nnc: 1', 'names: [person]', '']

yaml_content = '\n'.join(yaml_lines)
yaml_path = '/content/cdt_dataset/dataset.yaml'

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print('✅ dataset.yaml written:')
print('-' * 70)
print(yaml_content)
print('-' * 70)


# ── STEP 5: Verify label counts for train / val ──────────────────────────────
print('\n✔️  Verifying label counts post-sanitization...\n')

train_lbl_count = val_lbl_count = 0

print('Train:')
for seq in train_seqs:
    count = len(list((lbl_root / seq).glob('*.txt'))) if (lbl_root / seq).exists() else 0
    train_lbl_count += count
    flag = '' if count > 0 else ' ⚠️  EMPTY'
    print(f'  {seq}: {count} label files{flag}')

print('\nVal:')
for seq in val_seqs:
    count = len(list((lbl_root / seq).glob('*.txt'))) if (lbl_root / seq).exists() else 0
    val_lbl_count += count
    flag = '' if count > 0 else ' ⚠️  EMPTY'
    print(f'  {seq}: {count} label files{flag}')

print(f'\nTotal: {train_lbl_count} train labels, {val_lbl_count} val labels')

if train_lbl_count == 0:
    print('❌ No training labels found! Cannot proceed.')
    exit(1)


# ── STEP 6: Quick label sanity-check (sample 5 files) ───────────────────────
print('\n🔍 Quick label sanity-check (5 random samples)...\n')

import random
all_label_files = list(lbl_root.rglob('*.txt'))
samples = random.sample(all_label_files, min(5, len(all_label_files)))
for lf in samples:
    first_line = lf.read_text().splitlines()[0] if lf.read_text().strip() else '(empty)'
    print(f'  {lf.relative_to(lbl_root)}: {first_line}')


# ── STEP 7: Fine-tune YOLOv8n ────────────────────────────────────────────────
print('\n' + '=' * 70)
print('🚀 Starting YOLOv8n fine-tuning...')
print('=' * 70 + '\n')

model = YOLO('yolov8n.pt')

results = model.train(
    data          = yaml_path,
    epochs        = 50,
    imgsz         = 640,
    batch         = 16,
    lr0           = 0.001,
    lrf           = 0.01,
    device        = '0',
    project       = 'cdt_runs',
    name          = 'cdt_multidataset',
    save          = True,
    exist_ok      = True,
    patience      = 15,          # early-stop if no improvement for 15 epochs
    amp           = False,
    half          = False,
    workers       = 4,           # reduced to avoid DataLoader bottleneck on Colab
    close_mosaic  = 10,          # disable mosaic for last 10 epochs (stabilises training)
    label_smoothing = 0.0,
    plots         = True,
)

print('\n' + '=' * 70)
print('✅ Training complete!')
print('=' * 70)

# ── Metrics summary ──────────────────────────────────────────────────────────
if hasattr(results, 'results_dict'):
    rd = results.results_dict
    map50    = rd.get('metrics/mAP50(B)',    0)
    map50_95 = rd.get('metrics/mAP50-95(B)', 0)
    prec     = rd.get('metrics/precision(B)', 0)
    rec      = rd.get('metrics/recall(B)',    0)
    print(f'\n📊 Final Metrics:')
    print(f'   Precision : {prec:.4f}')
    print(f'   Recall    : {rec:.4f}')
    print(f'   mAP@0.50  : {map50:.4f}')
    print(f'   mAP@.5:.95: {map50_95:.4f}')


# ── STEP 8: Save best weights ────────────────────────────────────────────────
print('\n💾 Saving best weights...\n')

os.makedirs('/content/cdt/weights', exist_ok=True)
best_files = sorted(glob.glob('cdt_runs/cdt_multidataset*/weights/best.pt'))

if best_files:
    src = best_files[-1]
    dst = '/content/cdt/weights/yolov8n_cdt.pt'
    shutil.copy(src, dst)
    size_mb = Path(dst).stat().st_size / 1e6
    print(f'✅ Best model saved → {dst}  ({size_mb:.1f} MB)')
    print(f'   Source: {src}')
else:
    print('⚠️  No best.pt found — training may not have completed successfully')
    last_files = sorted(glob.glob('cdt_runs/cdt_multidataset*/weights/last.pt'))
    if last_files:
        shutil.copy(last_files[-1], '/content/cdt/weights/yolov8n_cdt_last.pt')
        print('   Falling back to last.pt checkpoint')

print('\n' + '=' * 70)
print('🎉 Pipeline complete!')
print('=' * 70)

In [ ]:
import torch

def apply_torch_patch():
    current_load = torch.load
    if hasattr(current_load, '__name__') and current_load.__name__ == 'patched_torch_load':
        return
    root_load = current_load
    while hasattr(root_load, '__wrapped__'):
        root_load = root_load.__wrapped__
        if hasattr(root_load, '__name__') and root_load.__name__ == 'patched_torch_load':
            return

    def patched_torch_load(*args, **kwargs):
        if 'weights_only' not in kwargs:
            kwargs['weights_only'] = False
        return root_load(*args, **kwargs)

    torch.load = patched_torch_load
    print('✅ PyTorch patch applied')

apply_torch_patch()
from ultralytics import YOLO

model = YOLO('cdt_runs/cdt_multidataset/weights/last.pt')
results = model.train(resume=True)

In [ ]:
# ── Save trained model to Google Drive (run this right after Cell 4) ──────────
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=True)

src = '/content/cdt/weights/yolov8n_cdt.pt'
dst = '/content/drive/MyDrive/cdt_project_data/cdt/weights/yolov8n_cdt.pt'

os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(src, dst)
print(f'✅ Model saved to Drive: {dst}')

In [ ]:
# ── Cell 5: Evaluate on UCF-CC-50 (density benchmark) ────────────────────────
import numpy as np
import cv2
from ultralytics import YOLO

model_path = '/content/cdt/weights/yolov8n_cdt.pt'
if not os.path.exists(model_path):
    model_path = 'yolov8n.pt'
    print('Using base YOLOv8n (fine-tuned weights not found)')

model = YOLO(model_path)
errors = []

print(f'{'Img':>4} | {'GT count':>8} | {'Pred count':>10} | {'MAE':>6}')
print('-' * 38)

for i in range(1, 51):
    img_path = f'/content/cdt_dataset/images/ucf_{i:02d}.jpg'
    dm_path  = f'/content/cdt_dataset/density/ucf_{i:02d}.npy'
    if not os.path.exists(img_path): continue
    img    = cv2.imread(img_path)
    gt_dm  = np.load(dm_path)
    gt_cnt = gt_dm.sum()
    res    = model(img, classes=[0], conf=0.35, verbose=False)
    pred   = len(res[0].boxes)
    mae    = abs(pred - gt_cnt)
    errors.append(mae)
    if i % 10 == 0:
        print(f'{i:>4} | {gt_cnt:>8.0f} | {pred:>10} | {mae:>6.1f}')

print('=' * 38)
print(f'Mean Absolute Error: {np.mean(errors):.2f} persons')
print(f'RMSE:                {np.sqrt(np.mean(np.array(errors)**2)):.2f} persons')
print()
print('Target: MAE < 50 for capstone | MAE < 20 for publication')

In [ ]:
# ── Cell 6: Prepare video source ─────────────────────────────────────────────
import os

os.makedirs('/content/cdt/videos', exist_ok=True)

# Option A: Use MOT17 sequence as demo video
SEQ  = '/content/datasets/MOT17/train/MOT17-09-FRCNN/img1'
OUT  = '/content/cdt/videos/mot17_demo.mp4'

if os.path.exists(SEQ) and not os.path.exists(OUT):
    os.system(f'ffmpeg -y -framerate 25 -pattern_type glob -i "{SEQ}/*.jpg" '
              f'-c:v libx264 -pix_fmt yuv420p {OUT} -loglevel quiet')
    print(f'✅ Demo video: {OUT}')
elif os.path.exists(OUT):
    print(f'✅ Demo video already exists: {OUT}')
else:
    print('⚠️  MOT17 not found. Upload your own .mp4 to /content/cdt/videos/')

# Option B: Upload your own video
# from google.colab import files
# uploaded = files.upload()   # select your .mp4
# import shutil
# shutil.move(list(uploaded.keys())[0], '/content/cdt/videos/my_video.mp4')
# VIDEO_SOURCE = '/content/cdt/videos/my_video.mp4'

VIDEO_SOURCE = OUT
print(f'VIDEO_SOURCE = {VIDEO_SOURCE}')

In [ ]:
# ── Cell 7: Upload project files from Google Drive ────────────────────────────
import os, shutil

os.makedirs('/content/cdt/static', exist_ok=True)
os.makedirs('/content/cdt/dataset/eth_ucy', exist_ok=True)

# Copy ETH/UCY data for mobility priors
if os.path.exists('/content/datasets/eth_ucy'):
    shutil.copytree('/content/datasets/eth_ucy',
                    '/content/cdt/dataset/eth_ucy',
                    dirs_exist_ok=True)
    print('✅ ETH/UCY copied to dataset/eth_ucy')

# OPTION A: Mount Google Drive and copy files
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy('/content/drive/MyDrive/cdt/main.py',         '/content/cdt/main.py')
# shutil.copy('/content/drive/MyDrive/cdt/static/index.html', '/content/cdt/static/index.html')

# OPTION B: Download from your GitHub repo (after pushing)
# REPO = 'https://raw.githubusercontent.com/YOUR_USERNAME/cdt/main'
# os.system(f'wget -q {REPO}/main.py -O /content/cdt/main.py')
# os.system(f'wget -q {REPO}/static/index.html -O /content/cdt/static/index.html')

print('Files present:')
for f in ['main.py', 'static/index.html', 'weights/yolov8n_cdt.pt']:
    path = f'/content/cdt/{f}'
    print(f'  {"✅" if os.path.exists(path) else "❌"} {f}')

In [ ]:
# ── Cell 8: Start real-time server + ngrok tunnel ────────────────────────────
import threading, os, time, sys
from pyngrok import ngrok

# Set your free ngrok authtoken from https://ngrok.com
NGROK_TOKEN  = '3B24xeUS4SvD78PxOdtcQwwCwj7_3rvFEdr1PrjA55at2sTEq'   # ← paste your token here
VIDEO_SOURCE = '/content/cdt/videos/mot17_demo.mp4'  # ← or your video

ngrok.set_auth_token(NGROK_TOKEN)
os.chdir('/content/cdt')

def run_server():
    sys.argv = ['main.py', '--source', VIDEO_SOURCE]
    exec(open('main.py').read(), {'__name__': '__main__'})

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(6)   # wait for server to bind

tunnel     = ngrok.connect(8000, 'http')
public_url = tunnel.public_url
ws_url     = public_url.replace('https','wss').replace('http','ws') + '/ws'

print('\n' + '='*60)
print('  CDT Dashboard is LIVE')
print(f'  Browser URL : {public_url}')
print(f'  WebSocket   : {ws_url}')
print(f'  REST API    : {public_url}/api/snapshot')
print('='*60)

In [ ]:
# ── Cell 9: Monitor pipeline in Colab output ─────────────────────────────────
import time, requests

print(f'{'Frame':>7} | {'Agents':>7} | {'Fused':>7} | {'Risk':>8} | Trend       | FPS   | Lat   | Alerts')
print('-' * 80)

for _ in range(60):
    try:
        r = requests.get('http://localhost:8000/api/snapshot', timeout=2)
        d = r.json()
        f  = d.get('fusion', {})
        print(
            f"{d.get('frame_idx',0):>7} | "
            f"{d.get('n_agents',0):>7} | "
            f"{d.get('fused_count',0):>7.1f} | "
            f"{d.get('risk_label','?'):>8} ({d.get('risk_score',0):.2f}) | "
            f"{d.get('crowd_trend','?'):<11} | "
            f"{d.get('fps',0):<5.1f} | "
            f"{d.get('latency_ms',0):<5.0f} | "
            f"{len(d.get('alerts',[]))} "
            f"[V:{f.get('video_count','?')} I:{f.get('iot_count','?')} F:{f.get('fused_count','?')}]"
        )
    except Exception as e:
        print(f'  Waiting... ({e})')
    time.sleep(2)

In [ ]:
# ── Cell 10: Save evaluation metrics and results ─────────────────────────────
import json, requests, numpy as np, os, time

# Collect 30 frames of pipeline data
samples = []
for _ in range(30):
    try:
        r = requests.get('http://localhost:8000/api/snapshot', timeout=2)
        samples.append(r.json())
    except: pass
    time.sleep(1)

if samples:
    fpss    = [s.get('fps',0) for s in samples]
    lats    = [s.get('latency_ms',0) for s in samples]
    risks   = [s.get('risk_score',0) for s in samples]
    confs   = [s.get('mean_confidence',0) for s in samples]
    fconfs  = [s.get('fusion',{}).get('confidence',0) for s in samples]

    print('='*50)
    print('EVALUATION METRICS — Paper Table')
    print('='*50)
    print(f'Mean FPS          : {np.mean(fpss):.1f}  (NFR: ≥10)  {"✅" if np.mean(fpss)>=10 else "❌"}')
    print(f'Mean latency      : {np.mean(lats):.0f} ms  (NFR: <2000)  {"✅" if np.mean(lats)<2000 else "❌"}')
    print(f'Max latency       : {np.max(lats):.0f} ms')
    print(f'Mean risk score   : {np.mean(risks):.3f}')
    print(f'Mean conf         : {np.mean(confs):.3f}')
    print(f'Mean fusion conf  : {np.mean(fconfs):.3f}')
    print()

    # Save to JSON
    metrics = {
        'mean_fps': round(float(np.mean(fpss)),2),
        'mean_latency_ms': round(float(np.mean(lats)),1),
        'nfr_fps_pass': bool(np.mean(fpss)>=10),
        'nfr_latency_pass': bool(np.mean(lats)<2000),
        'mean_risk_score': round(float(np.mean(risks)),3),
        'mean_confidence': round(float(np.mean(confs)),3),
        'mean_fusion_confidence': round(float(np.mean(fconfs)),3),
        'n_samples': len(samples),
    }

    # Fix: Ensure directory exists before saving
    os.makedirs('/content/cdt/outputs', exist_ok=True)

    with open('/content/cdt/outputs/pipeline_metrics.json','w') as f:
        json.dump(metrics, f, indent=2)
    print('Saved: /content/cdt/outputs/pipeline_metrics.json')